# Wald 检验统计量的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/simulation_wald_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对 H0: μ²=1 构造 Wald 统计量，模拟其在 H0（μ=-1）与 H1（μ=1.5）下的分布及拒绝率。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear`、`set more off`、`set seed 42`：准备模拟环境。本 notebook 研究 H0: μ²=1 这一**非线性约束**的 Wald 检验。


In [ ]:
%%stata
clear
set more off
set seed 42


**test statistic**

定义统计量计算程序：`wald_test_stat` 生成 n 个 N(μ,σ²) 样本，返回统计量 `chi2_stat = n(x̄²-1)²/(4x̄²·Var(x))`。
**推导思路**：把约束写成 g(μ)=μ²-1，由 delta 方法 √n[g(x̄)-g(μ)] →d N(0, g'(μ)²σ²)，其中 g'(μ)=2μ；把 μ 换成 x̄、σ² 换成样本方差（`r(Var)`），构造 Wald 统计量 n·g(x̄)²/[g'(x̄)²·s²]，即程序中的公式。在原假设下它渐近服从 χ²(1)。


In [ ]:
%%stata
cap program drop wald_test_stat
program define wald_test_stat, rclass
	syntax [,obs(integer 100) mu(real -1) sigma2(real 1)]
	clear
	tempvar x
	quietly{
		set obs `obs'
		gen `x'=rnormal(`mu',sqrt(`sigma2'))
		su `x'
		return scalar chi2_stat=`obs'*(r(mean)^2-1)^2/(4*r(mean)^2*r(Var))
	}
end


**upper tail of chi2(1)**

`local cv=invchi2tail(1,0.05)`：计算 χ²(1) 分布的 95% 分位数（约3.84）作为 5% 单侧检验的临界值，后面用 `xline` 画在图上。


In [ ]:
%%stata
local cv=invchi2tail(1,0.05)


**simulation under H0**

**H0 下的模拟**：`simulate chi2=r(chi2_stat), reps(5000)` 在默认参数 μ=-1（满足 μ²=1）下重复5000次；叠加 χ²(1) 密度曲线并画直方图，`xline(cv)` 标出临界值——直方图应与 χ²(1) 密度吻合；`gen p_005=chi2>=cv`、`su p_005` 计算实际拒绝率（应接近5%）。


In [ ]:
%%stata
simulate chi2=r(chi2_stat),reps(5000): wald_test_stat
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005


**simulation under H1**

**H1 下的模拟**：`clear` 后以 `mu(1.5)`（偏离原假设）重复5000次，同样画图。此时统计量的分布整体右移、超过临界值的比例大幅上升——`su p_005` 得到的拒绝率即该点的检验功效。两幅图对比直观展示了检验统计量在原假设与备择假设下分布的位置差异。


In [ ]:
%%stata
clear
simulate chi2=r(chi2_stat),reps(5000): wald_test_stat, mu(1.5)
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005
